In [1]:
# Loading environment variables from a .env file
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
# Creating an API client
from anthropic import Anthropic
import json

client = Anthropic()
model = "claude-haiku-4-5"

In [3]:
def add_user_message(messages: list, content: str) -> None:
    """
    Helper function to add a user message to the messages list.
    """
    messages.append({
        "role": "user",
        "content": content
    })
    # return messages

def add_assistant_message(messages: list, content: str) -> None:
    """
    Helper function to add an assistant message to the messages list.
    """
    messages.append({
        "role": "assistant",
        "content": content
    })
    # return messages


def chat(messages: list, model: str = "claude-haiku-4-5", max_tokens: int = 1000, system_prompt: str = None, temperature: float = 0.0, stop_sequences: list = None) -> str:
    """
    Function to send a chat request to the Claude API and return the assistant's response.
    """
    params = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages,
        "temperature": temperature
    }
    if system_prompt:
        params["system"] = system_prompt
    
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)
    return response.content[0].text

### Creating the Prompt Eval Dataset:

In [43]:
def generate_dataset():
    prompt = """
    Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

    Example output:
    ```json
    [
    {
        "task": "Description of task",
        "format": "python/json/regex",
        "solution_criteria": "What are the conditions for a solution to be considered correct?"
    },
    ...additional
    ]
    ```

    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
    * Focus on tasks that do not require writing much code    

    Please generate 3 objects.
    """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    response = chat(messages, model="claude-haiku-4-5", stop_sequences=["```"])
    return json.loads(response)

In [44]:
dataset = generate_dataset()

dataset

[{'task': "Parse an AWS S3 bucket name and region from an S3 URI (e.g., 's3://my-bucket/path/to/file') and return them as separate values",
  'format': 'python',
  'solution_criteria': 'The function should correctly extract the bucket name and return it. It should handle URIs with and without paths, and raise an error for invalid S3 URIs.'},
 {'task': 'Create a JSON configuration object for an AWS Lambda function that includes function name, runtime (Python 3.11), memory (256 MB), timeout (30 seconds), and an environment variable for a database connection string',
  'format': 'json',
  'solution_criteria': 'The JSON object should be valid and include all required fields with correct key names and appropriate data types matching AWS Lambda configuration standards.'},
 {'task': 'Write a regex pattern that matches valid AWS IAM role ARNs (format: arn:aws:iam::123456789012:role/RoleName)',
  'format': 'regex',
  'solution_criteria': 'The regex should match valid IAM role ARNs with 12-digit

In [45]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

### Creating the Eval Workflow (or Harness):

In [46]:
import re
import ast 

# Code Graders
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_by_code(test_case, output):
    if test_case["format"] == "json":
        return validate_json(output)
    elif test_case["format"] == "python":
        return validate_python(output)
    else:
        return validate_regex(output)

In [55]:
# Model Grader
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Original Task:
    <task>
    {test_case["task"]}
    </task>

    Solution:
    <solution>
    {output}
    </solution>

    Solution Criteria:
    <criteria>
    {test_case["solution_criteria"]}
    </criteria>
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [56]:
def run_prompt(task: str) -> str:
    """Run a prompt for a given task and return the response."""
    prompt = f"""
    Please provide a solution for the given task:

    {task}

    * The output should only be a Json, Python or Regex solution depending on the task requirement. Do not include any explanations or additional text.
    """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    response = chat(model="claude-haiku-4-5", messages=messages, stop_sequences=["```"])
    return response

In [57]:
def run_test_case(test_case: dict) -> dict:
    """Runs the run_prompt function with the given test case and returns the response with score"""
    response = run_prompt(test_case["task"])

    # Grade the response using the grade_by_model function
    evaluation = grade_by_model(test_case, response)
    model_score = evaluation.get("score", 0)
    reasoning = evaluation.get("reasoning", "")

    # Grade the response using the grade_by_code function
    code_score = grade_by_code(test_case, response)

    # Average the two scores for a final score
    score = (model_score + code_score) / 2

    return {
        "task": test_case["task"],
        "response": response,
        "code_score": code_score,
        "model_score": model_score,
        "score": score,
        "reasoning": reasoning
    }

In [58]:
def run_eval(dataset: list) -> list:
    """Runs the run_test_case function for each test case in the dataset and returns the results."""
    from statistics import mean
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean(result["score"] for result in results)
    print(f"Average Score: {average_score:.2f}")

    return results

In [59]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)


results = run_eval(dataset)

Average Score: 8.00


In [60]:
print(json.dumps(results, indent=2))

[
  {
    "task": "Parse an AWS S3 bucket name and region from an S3 URI (e.g., 's3://my-bucket/path/to/file') and return them as separate values",
    "response": "\nimport re\n\ndef parse_s3_uri(uri):\n    \"\"\"\n    Parse an S3 URI and extract bucket name and region.\n    \n    Args:\n        uri (str): S3 URI in format 's3://bucket-name/path' or 's3://bucket-name.region/path'\n    \n    Returns:\n        dict: Dictionary containing 'bucket' and 'region' keys\n    \"\"\"\n    # Pattern to match S3 URIs\n    pattern = r'^s3://([a-z0-9.-]+)(?:\\.([a-z0-9-]+))?(?:/.*)?$'\n    match = re.match(pattern, uri)\n    \n    if match:\n        bucket = match.group(1)\n        region = match.group(2) if match.group(2) else None\n        return {\n            'bucket': bucket,\n            'region': region\n        }\n    else:\n        raise ValueError(f\"Invalid S3 URI format: {uri}\")\n\n\n# Test cases\nif __name__ == \"__main__\":\n    test_uris = [\n        's3://my-bucket/path/to/file',\n